# Example 12.6.1 / 1: Check scenario mass before solving

Companion notebook for **12.6.1 Uncertainty and Scenario Design** in the AMPL lab Documentation, section "Check scenario mass before solving". It runs the page's example from top to bottom: the model, the amplpy way (data.py, run.py), then the pure-AMPL way (data.dat, model.run). Solver: HiGHS.

## Install Dependencies
💡 TIP: Visit [portal.ampl.com](https://portal.ampl.com/user/ampl/license/list) or [dev.ampl.com](https://dev.ampl.com/ampl/install.html) for AMPL setup on Colab, Docker, Python, Windows, Linux, macOS

In [ ]:
# Install dependencies
%pip install -q amplpy

# Google Colab & Kaggle integration
from amplpy import AMPL, ampl_notebook
ampl = ampl_notebook(
    modules=["highs"],      # modules to install
    license_uuid="default") # license to use

## Check scenario mass before solving

A `check` statement tests the probability mass, and a defined
parameter computes the expected demand per customer. `ampl.eval("check;")` is used because amplpy
has no method for the `check` command; `to_dict()` reads the result.

### model.mod
The model, written to a file that both ways of running it read.

In [2]:
%%writefile model.mod
set CUSTOMERS;                                        # customers served
set SCENARIOS;                                        # possible demand outcomes
param probability {SCENARIOS} >= 0;                   # scenario weight
param uncertain_demand {CUSTOMERS, SCENARIOS} >= 0;   # tons

# Scenario weights must form a probability distribution.
check: abs(sum {s in SCENARIOS} probability[s] - 1) <= 1e-9;

# Probability-weighted demand per customer.
param expected_demand {c in CUSTOMERS} =
    sum {s in SCENARIOS} probability[s] * uncertain_demand[c,s];

Writing model.mod


### amplpy: data.py and run.py

`ampl` is the `AMPL` object created above. Each run starts from a fresh session that has read `model.mod`; then `data.py` sets the data from Python and `run.py` solves and reads the results.

In [3]:
# A fresh session that has read model.mod, as data.py and run.py assume
ampl.reset()
ampl.eval("reset options;")  # reset keeps the options
ampl.read("model.mod")

In [4]:
# data.py
ampl.set["CUSTOMERS"] = ["east", "west"]
ampl.set["SCENARIOS"] = ["low", "base", "high"]
ampl.param["probability"] = {"low": 0.3, "base": 0.4, "high": 0.3}
ampl.param["uncertain_demand"] = {
    ("east", "low"): 80, ("east", "base"): 100, ("east", "high"): 130,
    ("west", "low"): 50, ("west", "base"): 60, ("west", "high"): 75,
}

In [5]:
# run.py
ampl.eval("check;")  # amplpy has no method for the check command
print(ampl.param["expected_demand"].to_dict())

{'east': 103, 'west': 61.5}


### Pure AMPL: data.dat and model.run

The same in pure AMPL, with the same `model.mod`; the `check` command runs the test and
`display` prints the expected demand:

In [6]:
%%writefile data.dat
set CUSTOMERS := east west;
set SCENARIOS := low base high;
param probability := low 0.3  base 0.4  high 0.3;
param uncertain_demand:  low  base  high :=
    east                  80   100   130
    west                  50    60    75;

Writing data.dat


In [7]:
%%ampl_eval
reset;
reset options;
model model.mod;
data data.dat;

In [8]:
%%ampl_eval
# model.run
check;
display expected_demand;

expected_demand [*] :=
east  103
west   61.5
;

